# Day 07: String Processing for Data Cleaning

**Dataset:** `Customer_Dataset.csv`

This notebook practices string operations, including trimming spaces, changing case, replacing
characters, splitting text, and extracting information, to build the text-cleaning skills
needed for real-world datasets. This dataset is intentionally messy: names in inconsistent
case, extra spaces, inconsistent gender/city labels, missing values, invalid emails, and
purchase amounts formatted as text (with `$` signs and commas).

## 1. Import Libraries and Load the Dataset

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("../datasets/Customer_Dataset.csv")

# Preview the data
df.head(10)

,Order ID,Customer Name,Age,Gender,City,Signup Date,Purchase Amount,Email
0,1000,Bob Lee,25.0,male,LA,01/22/2023,$45,alice@example.com
1,1001,Bob Lee,22.0,F,NaN,15-04-2023,$120.50,john@example.com
2,1002,MARY JOHNSON,-5.0,Female,miami,19-05-2023,$120.50,NaN
3,1003,Emma Brown,22.0,M,LA,15-04-2023,NaN,john@example.com
4,1004,David Kim,29.0,female,Chicago,01/22/2023,"1,200.00",bad-email
5,1005,Bob Lee,34.0,M,new york,2023.03.10,-20,NaN
6,1006,grace hall,25.0,F,miami,15-01-2023,300,john@example.com
7,1007,Paul Young,150.0,F,chicago,19-05-2023,"1,200.00",alice@example.com
8,1008,MARY JOHNSON,25.0,Female,Chicago,15-01-2023,"1,200.00",john@example.com
9,1009,Karen White,150.0,F,Boston,01-07-2023,-20,JANE@EXAMPLE.COM


In [ ]:
# Basic info about the dataset - notice the missing values in several columns
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order ID         43 non-null     int64  
 1   Customer Name    43 non-null     str    
 2   Age              39 non-null     float64
 3   Gender           37 non-null     str    
 4   City             41 non-null     str    
 5   Signup Date      40 non-null     str    
 6   Purchase Amount  37 non-null     str    
 7   Email            31 non-null     str    
dtypes: float64(1), int64(1), str(6)
memory usage: 2.8 KB


In [ ]:
# Count of missing values per column
df.isna().sum()

Order ID            0
Customer Name       0
Age                 4
Gender              6
City                2
Signup Date         3
Purchase Amount     6
Email              12
dtype: int64

**Pull the columns we'll clean into plain Python lists**

In [ ]:
names = list(df["Customer Name"])
genders = list(df["Gender"])
cities = list(df["City"])
emails = list(df["Email"])
purchase_amounts = list(df["Purchase Amount"])
signup_dates = list(df["Signup Date"])

print(names[:10])
print(genders[:10])
print(cities[:10])

['Bob Lee', 'Bob Lee', 'MARY JOHNSON', 'Emma Brown', ' David Kim', 'Bob Lee', 'grace hall', 'Paul Young', 'MARY JOHNSON', 'Karen White']
['male', 'F', 'Female', 'M', 'female', 'M', 'F', 'F', 'Female', 'F']
['LA', nan, 'miami ', 'LA', 'Chicago', 'new york', 'miami ', 'chicago', 'Chicago', 'Boston']


## 2. Common String Methods on Real Data

**`strip()` — remove leading/trailing whitespace**

In [ ]:
messy_name = names[4]     # " David Kim"
messy_city = cities[2]    # "miami " (with a trailing space)

print(f"Name before strip(): {repr(messy_name)}")
print(f"Name after strip():  {repr(messy_name.strip())}")

print(f"City before strip(): {repr(messy_city)}")
print(f"City after strip():  {repr(messy_city.strip())}")

Name before strip(): ' David Kim'
Name after strip():  'David Kim'
City before strip(): 'miami '
City after strip():  'miami'


**`lower()` and `upper()` — change case**

In [ ]:
messy_gender_values = ["male", "F", "Female", "M", "female"]

for value in messy_gender_values:
    print(f"{value!r:10} -> lower(): {value.lower()!r}")

'male'     -> lower(): 'male'
'F'        -> lower(): 'f'
'Female'   -> lower(): 'female'
'M'        -> lower(): 'm'
'female'   -> lower(): 'female'


**`replace()` — remove unwanted characters (cleaning `Purchase Amount`)**

In [ ]:
messy_amount = purchase_amounts[4]   # '1,200.00'
messy_amount_with_dollar = purchase_amounts[0]  # '$45'

cleaned_amount_1 = messy_amount.replace(",", "")
cleaned_amount_2 = messy_amount_with_dollar.replace("$", "")

print(f"{messy_amount!r} -> {cleaned_amount_1!r} -> {float(cleaned_amount_1)}")
print(f"{messy_amount_with_dollar!r} -> {cleaned_amount_2!r} -> {float(cleaned_amount_2)}")

'1,200.00' -> '1200.00' -> 1200.0
'$45' -> '45' -> 45.0


**`split()` — break inconsistent date formats into parts**

In [ ]:
date_slash = signup_dates[0]    # '01/22/2023'
date_dash = signup_dates[1]     # '15-04-2023'
date_dot = signup_dates[5]      # '2023.03.10'

print(date_slash.split("/"))
print(date_dash.split("-"))
print(date_dot.split("."))

['01', '22', '2023']
['15', '04', '2023']
['2023', '03', '10']


**Checking whether a string contains unwanted spaces**

In [ ]:
def has_unwanted_spaces(text):
    if pd.isna(text):
        return False
    return text != text.strip()


for value in cities[:6]:
    print(f"{value!r:15} -> has unwanted spaces: {has_unwanted_spaces(value)}")

'LA'            -> has unwanted spaces: False
nan             -> has unwanted spaces: False
'miami '        -> has unwanted spaces: True
'LA'            -> has unwanted spaces: False
'Chicago'       -> has unwanted spaces: False
'new york'      -> has unwanted spaces: False


**Extracting information: pulling just the numeric part out of every real `Purchase Amount`**

In [ ]:
def extract_amount(raw_amount):
    if pd.isna(raw_amount):
        return None
    return float(raw_amount.replace("$", "").replace(",", ""))


extracted_amounts = [extract_amount(amount) for amount in purchase_amounts[:10]]
print(extracted_amounts)

[45.0, 120.5, 120.5, None, 1200.0, -20.0, 300.0, 1200.0, 1200.0, -20.0]


## 3. A Small Text-Cleaning Function

A single reusable function that trims whitespace and standardizes case, which are the two most common
first steps in cleaning any text field. It also safely handles missing values.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return None
    return value.strip()


# Test it on a couple of deliberately messy real values
print(repr(clean_text(names[4])))    # " David Kim" -> "David Kim"
print(repr(clean_text(cities[2])))   # "miami " -> "miami"
print(repr(clean_text(cities[1])))   # NaN -> None

'David Kim'
'miami'
None


## 4. Cleaning Names, Emails, and Categories

**Cleaning names — trim spaces and convert to title case**

In [ ]:
cleaned_names = [clean_text(name).title() if clean_text(name) else None for name in names[:10]]
print(cleaned_names)

['Bob Lee', 'Bob Lee', 'Mary Johnson', 'Emma Brown', 'David Kim', 'Bob Lee', 'Grace Hall', 'Paul Young', 'Mary Johnson', 'Karen White']


**Cleaning emails — trim spaces, convert to lowercase, and flag invalid values**

In [ ]:
def clean_email(raw_email):
    if pd.isna(raw_email) or raw_email.strip() == "":
        return None

    email = raw_email.strip().lower()

    if "@" not in email:
        return None   # not a valid-looking email, e.g. "bad-email"

    return email


for email in emails[:10]:
    print(f"{email!r:30} -> {clean_email(email)!r}")

'alice@example.com '           -> 'alice@example.com'
'john@example.com'             -> 'john@example.com'
nan                            -> None
'john@example.com'             -> 'john@example.com'
'bad-email'                    -> None
nan                            -> None
'john@example.com'             -> 'john@example.com'
'alice@example.com '           -> 'alice@example.com'
'john@example.com'             -> 'john@example.com'
'JANE@EXAMPLE.COM'             -> 'jane@example.com'


**Standardizing categories — `Gender` and `City`**

Trim and lowercase each raw value first, then map it to one consistent, standardized label.

In [ ]:
def standardize_gender(raw_gender):
    if pd.isna(raw_gender):
        return None

    normalized = raw_gender.strip().lower()
    gender_map = {
        "m": "Male",
        "male": "Male",
        "f": "Female",
        "female": "Female"
    }
    return gender_map.get(normalized, "Unknown")


cleaned_genders = [standardize_gender(gender) for gender in genders[:10]]
print(cleaned_genders)

['Male', 'Female', 'Female', 'Male', 'Female', 'Male', 'Female', 'Female', 'Female', 'Female']


In [ ]:
def standardize_city(raw_city):
    if pd.isna(raw_city):
        return None

    normalized = raw_city.strip().lower()
    city_map = {
        "la": "Los Angeles",
        "los angeles": "Los Angeles",
        "new york": "New York",
        "chicago": "Chicago",
        "miami": "Miami",
        "boston": "Boston"
    }
    return city_map.get(normalized, raw_city.strip().title())


cleaned_cities = [standardize_city(city) for city in cities[:10]]
print(cleaned_cities)

['Los Angeles', None, 'Miami', 'Los Angeles', 'Chicago', 'New York', 'Miami', 'Chicago', 'Chicago', 'Boston']


**Putting it all together: building fully cleaned customer records**

In [ ]:
clean_customers = []

for i in range(10):
    clean_customers.append({
        "name": clean_text(names[i]).title() if clean_text(names[i]) else None,
        "email": clean_email(emails[i]),
        "gender": standardize_gender(genders[i]),
        "city": standardize_city(cities[i]),
        "purchase_amount": extract_amount(purchase_amounts[i])
    })

for customer in clean_customers:
    print(customer)

{'name': 'Bob Lee', 'email': 'alice@example.com', 'gender': 'Male', 'city': 'Los Angeles', 'purchase_amount': 45.0}
{'name': 'Bob Lee', 'email': 'john@example.com', 'gender': 'Female', 'city': None, 'purchase_amount': 120.5}
{'name': 'Mary Johnson', 'email': None, 'gender': 'Female', 'city': 'Miami', 'purchase_amount': 120.5}
{'name': 'Emma Brown', 'email': 'john@example.com', 'gender': 'Male', 'city': 'Los Angeles', 'purchase_amount': None}
{'name': 'David Kim', 'email': None, 'gender': 'Female', 'city': 'Chicago', 'purchase_amount': 1200.0}
{'name': 'Bob Lee', 'email': None, 'gender': 'Male', 'city': 'New York', 'purchase_amount': -20.0}
{'name': 'Grace Hall', 'email': 'john@example.com', 'gender': 'Female', 'city': 'Miami', 'purchase_amount': 300.0}
{'name': 'Paul Young', 'email': 'alice@example.com', 'gender': 'Female', 'city': 'Chicago', 'purchase_amount': 1200.0}
{'name': 'Mary Johnson', 'email': 'john@example.com', 'gender': 'Female', 'city': 'Chicago', 'purchase_amount': 1200.0

## Outcome

This notebook practiced core string-cleaning operations, namely, `strip()`, `lower()`, `upper()`,
`replace()`, and `split()`, entirely on the real, intentionally messy `Customer_Dataset.csv`
file. I wrote a `has_unwanted_spaces()` check, a reusable `clean_text()` function that safely
handles missing values while trimming whitespace, an `extract_amount()` function that strips
`$` signs and commas out of purchase amounts and converts them to numbers, a `clean_email()`
function that validates and lowercases emails, and `standardize_gender()` /
`standardize_city()` functions that map inconsistent category text (`"M"`, `"male"`, `"LA"`,
`"new york"`) down to one consistent, standardized label. This showed how a handful of simple
string operations, combined with basic missing-value handling, can turn real, messy exported
data into clean, consistent records ready for further analysis.